# Conheça o Colab

In [ ]:
# -*- coding: utf-8 -*-
"""
Atividade: Processar PDFs da Aula 2 com Docling
Conversão de 3 PDFs para Markdown
"""

import os
import subprocess
import sys
import zipfile
import requests
from pathlib import Path
from google.colab import files
from docling.document_converter import DocumentConverter
import logging

# Configurar logging para ver o progresso
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ==================== CONFIGURAÇÃO ====================
PASTA_AULA = "aula_2"
URLS_PDFS = {
    "bioetica_e_ia.pdf": "https://drive.google.com/uc?export=download&id=SEU_ID_1",
    "escrita_academica_ia.pdf": "https://drive.google.com/uc?export=download&id=SEU_ID_2",
    "twitter_algoritmo.pdf": "https://drive.google.com/uc?export=download&id=SEU_ID_3"
}

# ==================== FUNÇÕES AUXILIARES ====================

def criar_pasta_aula():
    """Cria a pasta aula_2 se não existir"""
    if not os.path.exists(PASTA_AULA):
        os.makedirs(PASTA_AULA)
        logger.info(f" Pasta '{PASTA_AULA}' criada com sucesso!")
    else:
        logger.info(f" Pasta '{PASTA_AULA}' já existe.")

def baixar_pdfs_do_drive():
    """
    Faz o upload dos PDFs manualmente (via Colab) para garantir que estão no ambiente
    Esta função substitui o download direto do Drive que pode ter problemas de autenticação
    """
    print("\n" + "="*60)
    print(" FAÇA O UPLOAD DOS 3 PDFs")
    print("="*60)
    print("Por favor, faça o upload dos seguintes arquivos:")
    print("  1. bioetica_e_ia.pdf")
    print("  2. escrita_academica_ia.pdf")
    print("  3. twitter_algoritmo.pdf")
    print("="*60 + "\n")

    arquivos_baixados = []

    for i in range(1, 4):
        print(f"\n Documento {i}/3 - Clique em 'Escolher arquivo' e selecione o PDF")
        uploaded = files.upload()

        for nome_arquivo, conteudo in uploaded.items():
            # Salvar o arquivo na pasta aula_2
            caminho_destino = os.path.join(PASTA_AULA, nome_arquivo)
            with open(caminho_destino, 'wb') as f:
                f.write(conteudo)
            arquivos_baixados.append(caminho_destino)
            logger.info(f" Arquivo salvo: {caminho_destino}")

    return arquivos_baixados

def verificar_pdfs_baixados():
    """Verifica quais PDFs estão disponíveis na pasta aula_2"""
    pdfs_encontrados = []
    for arquivo in os.listdir(PASTA_AULA):
        if arquivo.endswith('.pdf'):
            caminho = os.path.join(PASTA_AULA, arquivo)
            pdfs_encontrados.append(caminho)
            logger.info(f"📄 PDF encontrado: {arquivo} ({os.path.getsize(caminho)} bytes)")
    return pdfs_encontrados

# ==================== CONVERSÃO PARA MARKDOWN ====================

def converter_pdf_para_markdown(caminho_pdf):
    """
    Converte um único PDF para Markdown usando Docling
    Retorna o caminho do arquivo .md gerado
    """
    try:
        nome_base = os.path.splitext(os.path.basename(caminho_pdf))[0]
        caminho_md = os.path.join(PASTA_AULA, f"{nome_base}.md")

        logger.info(f" Convertendo: {nome_base}.pdf")

        # Inicializar o conversor do Docling
        converter = DocumentConverter()

        # Realizar a conversão
        resultado = converter.convert(caminho_pdf)

        # Extrair o conteúdo em Markdown
        markdown_texto = resultado.document.export_to_markdown()

        # Salvar o arquivo Markdown
        with open(caminho_md, 'w', encoding='utf-8') as f:
            f.write(markdown_texto)

        logger.info(f" Convertido: {nome_base}.md")
        return caminho_md

    except Exception as e:
        logger.error(f" Erro ao converter {caminho_pdf}: {str(e)}")
        return None

def converter_todos_pdfs():
    """Converte todos os PDFs da pasta aula_2 para Markdown"""
    pdfs = verificar_pdfs_baixados()

    if not pdfs:
        logger.warning(" Nenhum PDF encontrado na pasta 'aula_2'.")
        return []

    logger.info(f"\n Iniciando conversão de {len(pdfs)} PDFs...")
    arquivos_md = []

    for pdf in pdfs:
        md = converter_pdf_para_markdown(pdf)
        if md:
            arquivos_md.append(md)

    return arquivos_md

# ==================== RELATÓRIO E FINALIZAÇÃO ====================

def gerar_relatorio(arquivos_md):
    """Gera um relatório com informações sobre os arquivos convertidos"""
    if not arquivos_md:
        print("\n Nenhum arquivo foi convertido para Markdown.")
        return

    print("\n" + "="*60)
    print(" RELATÓRIO DE CONVERSÃO")
    print("="*60)

    for i, arquivo in enumerate(arquivos_md, 1):
        if os.path.exists(arquivo):
            with open(arquivo, 'r', encoding='utf-8') as f:
                conteudo = f.read()
                palavras = len(conteudo.split())
                caracteres = len(conteudo)

                print(f"\n Documento {i}: {os.path.basename(arquivo)}")
                print(f"   Palavras: {palavras:,}")
                print(f"    Caracteres: {caracteres:,}")
                print(f"   Tamanho: {os.path.getsize(arquivo):,} bytes")

    print("\n" + "="*60)
    print(f" Total de arquivos convertidos: {len(arquivos_md)}")
    print("="*60)

def baixar_resultados():
    """Cria um ZIP com os arquivos Markdown e disponibiliza para download"""
    arquivos_md = [f for f in os.listdir(PASTA_AULA) if f.endswith('.md')]

    if not arquivos_md:
        print(" Nenhum arquivo Markdown encontrado para baixar.")
        return

    nome_zip = "aula_2_markdowns.zip"
    caminho_zip = os.path.join(PASTA_AULA, nome_zip)

    with zipfile.ZipFile(caminho_zip, 'w') as zipf:
        for arquivo in arquivos_md:
            caminho_completo = os.path.join(PASTA_AULA, arquivo)
            zipf.write(caminho_completo, arquivo)
            print(f" Adicionado ao ZIP: {arquivo}")

    print(f"\n Baixando o arquivo ZIP: {nome_zip}")
    files.download(caminho_zip)

# ==================== FUNÇÃO PRINCIPAL ====================

def main():
    """Função principal que executa toda a atividade"""
    print(" ATIVIDADE: PROCESSAR PDFS DA AULA 2 COM DOCLING")
    print("="*60)

    try:
        # 1. Criar a pasta aula_2
        criar_pasta_aula()

        # 2. Baixar os PDFs (via upload manual no Colab)
        print("\n Etapa 1: Upload dos PDFs")
        print("-" * 40)
        pdfs_baixados = baixar_pdfs_do_drive()

        if not pdfs_baixados:
            print("❌ Nenhum PDF foi enviado. Encerrando...")
            return

        # 3. Verificar os PDFs na pasta
        print("\n Etapa 2: Verificando PDFs")
        print("-" * 40)
        pdfs = verificar_pdfs_baixados()

        if len(pdfs) != 3:
            print(f" Atenção: Foram encontrados {len(pdfs)} PDFs, mas esperava 3.")
            print("   Verifique se todos os arquivos foram enviados corretamente.")

        # 4. Converter para Markdown
        print("\n Etapa 3: Convertendo PDFs para Markdown")
        print("-" * 40)
        arquivos_md = converter_todos_pdfs()

        # 5. Gerar relatório
        gerar_relatorio(arquivos_md)

        # 6. Baixar os resultados
        print("\n Etapa 4: Download dos resultados")
        print("-" * 40)
        baixar_resultados()

        print("\n" + "="*60)
        print(" ATIVIDADE CONCLUÍDA COM SUCESSO!")
        print("="*60)

    except Exception as e:
        print(f"\n ERRO: {str(e)}")
        print("Por favor, verifique os arquivos e tente novamente.")

# ==================== EXECUÇÃO ====================
if __name__ == "__main__":
    main()

 ATIVIDADE: PROCESSAR PDFS DA AULA 2 COM DOCLING

 Etapa 1: Upload dos PDFs
----------------------------------------

 FAÇA O UPLOAD DOS 3 PDFs
Por favor, faça o upload dos seguintes arquivos:
  1. bioetica_e_ia.pdf
  2. escrita_academica_ia.pdf
  3. twitter_algoritmo.pdf


 Documento 1/3 - Clique em 'Escolher arquivo' e selecione o PDF


Saving twitter_algoritmo.pdf to twitter_algoritmo (2).pdf

 Documento 2/3 - Clique em 'Escolher arquivo' e selecione o PDF


Saving escrita_academica_ia.pdf to escrita_academica_ia (2).pdf

 Documento 3/3 - Clique em 'Escolher arquivo' e selecione o PDF


[INFO] 2026-08-05 18:16:23,720 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-05 18:16:23,723 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-05 18:16:23,734 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-08-05 18:16:23,736 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth


Saving bioetica_e_ia.pdf to bioetica_e_ia (2).pdf

 Etapa 2: Verificando PDFs
----------------------------------------
 Atenção: Foram encontrados 6 PDFs, mas esperava 3.
   Verifique se todos os arquivos foram enviados corretamente.

 Etapa 3: Convertendo PDFs para Markdown
----------------------------------------


...

ERROR:docling.datamodel.document:Failed to resolve input source 'aula_2/bioetica_e_ia (1).pdf': [Errno 2] No such file or directory: 'aula_2/bioetica_e_ia (1).pdf'
ERROR:__main__: Erro ao converter aula_2/bioetica_e_ia (1).pdf: Conversion failed for: bioetica_e_ia (1).pdf with status: failure. Errors: [Errno 2] No such file or directory: 'aula_2/bioetica_e_ia (1).pdf'
ERROR:docling.datamodel.document:Failed to resolve input source 'aula_2/twitter_algoritmo (1).pdf': [Errno 2] No such file or directory: 'aula_2/twitter_algoritmo (1).pdf'
ERROR:__main__: Erro ao converter aula_2/twitter_algoritmo (1).pdf: Conversion failed for: twitter_algoritmo (1).pdf with status: failure. Errors: [Errno 2] No such file or directory: 'aula_2/twitter_algoritmo (1).pdf'



 RELATÓRIO DE CONVERSÃO

 Documento 1: twitter_algoritmo (2).md
   Palavras: 7,828
    Caracteres: 54,440
   Tamanho: 56,135 bytes

 Documento 2: escrita_academica_ia (1).md
   Palavras: 6,108
    Caracteres: 42,678
   Tamanho: 43,754 bytes

 Documento 3: bioetica_e_ia (2).md
   Palavras: 7,354
    Caracteres: 51,213
   Tamanho: 52,745 bytes

 Documento 4: escrita_academica_ia (2).md
   Palavras: 6,108
    Caracteres: 42,678
   Tamanho: 43,754 bytes

 Total de arquivos convertidos: 4

 Etapa 4: Download dos resultados
----------------------------------------
 Adicionado ao ZIP: bioetica_e_ia (2).md
 Adicionado ao ZIP: escrita_academica_ia (2).md
 Adicionado ao ZIP: twitter_algoritmo (2).md
 Adicionado ao ZIP: escrita_academica_ia (1).md

 Baixando o arquivo ZIP: aula_2_markdowns.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 ATIVIDADE CONCLUÍDA COM SUCESSO!
